# Tutorial: MLflow for Experiment Tracking and Model Registry

This notebook shows how to use MLflow to track experiments for:
- a simple scikit-learn model
- a PyTorch model
- model registry operations (versions, aliases, loading)

Both model workflows use the census dataset prepared by `process_census_data.py`.

## 1) Install and import

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from datetime import datetime

import mlflow
import mlflow.pytorch
import mlflow.sklearn
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from mlflow.tracking import MlflowClient
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

try:
    import skops.io  # noqa: F401
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "skops"])
    import skops.io  # noqa: F401

SKLEARN_SERIALIZATION_FORMAT = "skops"

try:
    from exercises.process_census_data import prepare_census_data
except ModuleNotFoundError:
    from solutions.process_census_data import prepare_census_data

c:\Users\NC3769\Code\AI_methods_in_economics_and_finance_research\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2) Configure MLflow tracking

In [2]:
MLFLOW_DIR = PROJECT_ROOT / "mlruns_tutorial"
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
TRACKING_URI = MLFLOW_DIR.as_uri()

In [ ]:
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_registry_uri(TRACKING_URI)

In [ ]:
EXPERIMENT_NAME = "mlflow_tutorial_experiment"
mlflow.set_experiment(EXPERIMENT_NAME)

c:\Users\NC3769\Code\AI_methods_in_economics_and_finance_research\.venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/24 14:21:24 INFO mlflow.tracking.fluent: Experiment with name 'mlflow_tutorial_experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///C:/Users/NC3769/Code/AI_methods_in_economics_and_finance_research/mlruns_tutorial/423036649745527009', creation_time=1779625284382, experiment_id='423036649745527009', last_update_time=1779625284382, lifecycle_stage='active', name='mlflow_tutorial_experiment', tags={}, trace_location=None, workspace='default'>

In [7]:
client = MlflowClient()

## 3) Track a simple scikit-learn model (census data)

In [8]:
X_train, X_test, y_train, y_test = prepare_census_data(
    path=str(PROJECT_ROOT / "data" / "census-income.data"),
    random_state=42,
    rebalance=False,
    split_validation=False,
 )

In [9]:
X_train.shape, X_test.shape

((149642, 126), (49881, 126))

In [10]:
sklearn_configs = [
    {"C": 0.1, "max_iter": 300},
    {"C": 1.0, "max_iter": 500},
    {"C": 5.0, "max_iter": 700},
]

In [ ]:
sklearn_run_ids = []
sklearn_results = []

for cfg in sklearn_configs:
    simple_model = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "clf",
                LogisticRegression(
                    max_iter=cfg["max_iter"],
                    C=cfg["C"],
                    random_state=42,
                ),
            ),
        ]
    )

    with mlflow.start_run(run_name=f"sklearn_logreg_C_{cfg['C']}") as run:
        simple_model.fit(X_train, y_train)
        preds = simple_model.predict(X_test)
        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds)

        mlflow.log_param("model_type", "logistic_regression")
        mlflow.log_param("C", cfg["C"] )
        mlflow.log_param("max_iter", cfg["max_iter"] )
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1", f1)
        mlflow.sklearn.log_model(
            simple_model,
            name="simple_model",
            serialization_format=SKLEARN_SERIALIZATION_FORMAT,
        )

        sklearn_run_ids.append(run.info.run_id)
        sklearn_results.append({"run_id": run.info.run_id, "C": cfg["C"], "max_iter": cfg["max_iter"], "accuracy": acc, "f1": f1})

2026/05/24 14:21:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/24 14:21:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/24 14:21:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/24 14:21:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

In [12]:
sklearn_results_df = pd.DataFrame(sklearn_results).sort_values("f1", ascending=False).reset_index(drop=True)
best_sklearn_run_id = sklearn_results_df.loc[0, "run_id"]
sklearn_run_id = best_sklearn_run_id
sklearn_results_df

,run_id,C,max_iter,accuracy,f1
0,0dcc01e8d5b84bf884bb9ec7a83db14a,5.0,700,0.952327,0.448516
1,9d22340a592a417a84bb0585f15aed23,1.0,500,0.952367,0.448468
2,a7e90437a8e94bc6b8c8f50debf316de,0.1,300,0.952367,0.447185


## 4) Track a PyTorch model (census data)

In [13]:
torch.manual_seed(42)

In [14]:
X_train_t = torch.from_numpy(X_train.to_numpy(dtype="float32"))

In [15]:
y_train_t = torch.from_numpy(y_train.to_numpy(dtype="float32")).view(-1, 1)

In [16]:
X_test_t = torch.from_numpy(X_test.to_numpy(dtype="float32"))

In [17]:
y_test_t = torch.from_numpy(y_test.to_numpy(dtype="float32")).view(-1, 1)

In [18]:
torch_configs = [
    {"hidden_units": 8, "lr": 0.01, "epochs": 30},
    {"hidden_units": 16, "lr": 0.01, "epochs": 40},
    {"hidden_units": 32, "lr": 0.005, "epochs": 50},
]

In [19]:
torch_run_ids = []
torch_results = []

In [20]:
len(torch_configs)

3

In [21]:
torch_configs

[{'hidden_units': 8, 'lr': 0.01, 'epochs': 30},
 {'hidden_units': 16, 'lr': 0.01, 'epochs': 40},
 {'hidden_units': 32, 'lr': 0.005, 'epochs': 50}]

In [ ]:
for cfg in torch_configs:
    torch.manual_seed(42)
    torch_model = nn.Sequential(
        nn.Linear(X_train.shape[1], cfg["hidden_units"]),
        nn.ReLU(),
        nn.Linear(cfg["hidden_units"], 1),
        nn.Sigmoid(),
    )
    criterion = nn.BCELoss()
    optimizer = optim.Adam(torch_model.parameters(), lr=cfg["lr"] )

    with mlflow.start_run(run_name=f"torch_mlp_h{cfg['hidden_units']}_lr{cfg['lr']}") as run:
        mlflow.log_param("model_type", "torch_mlp")
        mlflow.log_param("hidden_units", cfg["hidden_units"] )
        mlflow.log_param("lr", cfg["lr"] )
        mlflow.log_param("epochs", cfg["epochs"] )

        for epoch in range(cfg["epochs"]):
            torch_model.train()
            optimizer.zero_grad()
            outputs = torch_model(X_train_t)
            loss = criterion(outputs, y_train_t)
            loss.backward()
            optimizer.step()
            if (epoch + 1) % 10 == 0:
                mlflow.log_metric("train_loss", float(loss.item()), step=epoch + 1)

        torch_model.eval()
        with torch.no_grad():
            test_probs = torch_model(X_test_t)
            test_preds = (test_probs >= 0.5).float()
            test_acc = (test_preds.eq(y_test_t)).float().mean().item()

        mlflow.log_metric("test_accuracy", test_acc)
        mlflow.pytorch.log_model(torch_model, name="torch_model")

        torch_run_ids.append(run.info.run_id)
        torch_results.append({
            "run_id": run.info.run_id,
            "hidden_units": cfg["hidden_units"],
            "lr": cfg["lr"],
            "epochs": cfg["epochs"],
            "test_accuracy": test_acc,
        })

2026/05/24 14:22:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/24 14:22:09 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/05/24 14:22:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/24 14:22:21 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/05/

In [23]:
torch_results_df = pd.DataFrame(torch_results).sort_values("test_accuracy", ascending=False).reset_index(drop=True)
best_torch_run_id = torch_results_df.loc[0, "run_id"]
torch_run_id = best_torch_run_id
torch_results_df

,run_id,hidden_units,lr,epochs,test_accuracy
0,e6c9189c44074a0293d9f689e6855ba4,16,0.010,40,0.940939
1,d432d1626c014d939c08928e0e4d1678,32,0.005,50,0.935486
2,c93dd997e4be4dda919e7f411ff6049e,8,0.010,30,0.929993


## 5) Compare runs

In [24]:
runs_df = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])

In [25]:
cols = [
    "run_id",
    "tags.mlflow.runName",
    "params.model_type",
    "params.C",
    "params.max_iter",
    "params.hidden_units",
    "params.lr",
    "params.epochs",
    "metrics.accuracy",
    "metrics.f1",
    "metrics.test_accuracy",
]

In [26]:
runs_df[cols].fillna("-")

,run_id,tags.mlflow.runName,params.model_type,params.C,params.max_iter,params.hidden_units,params.lr,params.epochs,metrics.accuracy,metrics.f1,metrics.test_accuracy
0,d432d1626c014d939c08928e0e4d1678,torch_mlp_h32_lr0.005,torch_mlp,-,-,32,0.005,50,-,-,0.935486
1,e6c9189c44074a0293d9f689e6855ba4,torch_mlp_h16_lr0.01,torch_mlp,-,-,16,0.01,40,-,-,0.940939
2,c93dd997e4be4dda919e7f411ff6049e,torch_mlp_h8_lr0.01,torch_mlp,-,-,8,0.01,30,-,-,0.929993
3,0dcc01e8d5b84bf884bb9ec7a83db14a,sklearn_logreg_C_5.0,logistic_regression,5.0,700,-,-,-,0.952327,0.448516,-
4,9d22340a592a417a84bb0585f15aed23,sklearn_logreg_C_1.0,logistic_regression,1.0,500,-,-,-,0.952367,0.448468,-
5,a7e90437a8e94bc6b8c8f50debf316de,sklearn_logreg_C_0.1,logistic_regression,0.1,300,-,-,-,0.952367,0.447185,-


## 6) Register models in Model Registry

In [27]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

In [28]:
simple_registered_name = f"tutorial_simple_model_{timestamp}"

In [29]:
torch_registered_name = f"tutorial_torch_model_{timestamp}"

In [30]:
simple_uri = f"runs:/{sklearn_run_id}/simple_model"

In [31]:
torch_uri = f"runs:/{torch_run_id}/torch_model"

In [32]:
simple_version = mlflow.register_model(model_uri=simple_uri, name=simple_registered_name)

c:\Users\NC3769\Code\AI_methods_in_economics_and_finance_research\.venv\Lib\site-packages\mlflow\tracking\_model_registry\utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)
Successfully registered model 'tutorial_simple_model_20260524_142241'.
2026/05/24 14:22:41 WARNING mlflow.tracking._model_registry.fluent: Run with id 0dcc01e8d5b84bf884bb9ec7a83db14a has no artifacts at artifact path 'simple_model', registering model based on models:/m-c13fe97d20544310922712f5b9c3adc5 instead
Created version '1' of model 'tutorial_simple_model_20260524_142241'.


In [33]:
torch_version = mlflow.register_model(model_uri=torch_uri, name=torch_registered_name)

Successfully registered model 'tutorial_torch_model_20260524_142241'.
2026/05/24 14:22:42 WARNING mlflow.tracking._model_registry.fluent: Run with id e6c9189c44074a0293d9f689e6855ba4 has no artifacts at artifact path 'torch_model', registering model based on models:/m-c20df03b064e4fffa25e0ea5e99e5a71 instead
Created version '1' of model 'tutorial_torch_model_20260524_142241'.


In [34]:
simple_version.version, torch_version.version

(1, 1)

## 7) Add aliases (champion/candidate)

In [35]:
client.set_registered_model_alias(simple_registered_name, "champion", simple_version.version)

In [36]:
client.set_registered_model_alias(torch_registered_name, "candidate", torch_version.version)

## 8) Load models from registry

In [37]:
loaded_simple = mlflow.sklearn.load_model(f"models:/{simple_registered_name}@champion")

In [38]:
loaded_torch = mlflow.pytorch.load_model(f"models:/{torch_registered_name}@candidate")

In [39]:
loaded_simple.predict(X_test[:5])

array([0, 0, 0, 0, 0])

In [40]:
loaded_torch.eval()

Sequential(
  (0): Linear(in_features=126, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=1, bias=True)
  (3): Sigmoid()
)

In [41]:
with torch.no_grad():
    loaded_probs = loaded_torch(X_test_t[:5])
loaded_probs

tensor([[2.8091e-02],
        [8.5818e-02],
        [4.1000e-22],
        [4.3296e-01],
        [2.0178e-03]])

## 9) Inspect registry entries

In [42]:
client.get_registered_model(simple_registered_name).aliases

{'champion': '1'}

In [43]:
client.get_registered_model(torch_registered_name).aliases

{'candidate': '1'}

## 10) Open the MLflow UI

In [44]:
mlflow_ui_cmd = [
    "mlflow",
    "ui",
    "--backend-store-uri",
    TRACKING_URI,
    "--host",
    "127.0.0.1",
    "--port",
    "5000",
]

if os.name == "nt":
    mlflow_ui_process = subprocess.Popen(
        mlflow_ui_cmd,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        creationflags=subprocess.CREATE_NEW_PROCESS_GROUP | subprocess.DETACHED_PROCESS,
    )
else:
    mlflow_ui_process = subprocess.Popen(
        mlflow_ui_cmd,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )

print(f"MLflow UI started at http://127.0.0.1:5000 using store {TRACKING_URI}")
mlflow_ui_process.pid

MLflow UI started at http://127.0.0.1:5000 using store file:///C:/Users/NC3769/Code/AI_methods_in_economics_and_finance_research/mlruns_tutorial


14444

In [45]:
MLFLOW_DIR

WindowsPath('C:/Users/NC3769/Code/AI_methods_in_economics_and_finance_research/mlruns_tutorial')

Run the previous code cell to start the MLflow UI server in the background.

If running from a terminal, use a URI (not a raw Windows path):

`mlflow ui --backend-store-uri file:///C:/Users/NC3769/Code/AI_methods_in_economics_and_finance_research/mlruns_tutorial`

Then open http://127.0.0.1:5000 to browse experiments and registry.